In [3]:
import pandas as pd

df = pd.read_parquet('말뭉치_2024.parquet.gzip')
print(f'전체 형태소 행: {len(df):,}')
print(f'전체 문장 수: {df.groupby(["표본 번호","문장"]).ngroups:,}')

전체 형태소 행: 2,608,912
전체 문장 수: 235,902


In [4]:
# 오류가 있는 행만 필터
err_rows = df[df['오류 양상'] != '0'].copy()
print(f'오류 행 수: {len(err_rows):,}')

오류 행 수: 137,298


In [1]:
def make_error_signature(row):
    loc = row["오류 위치"]
    pat = row["오류 양상"]
    lvl = row["오류 층위"] if row["오류 층위"] != "0" else ""
    if lvl:
        return f"{loc}:{pat}:{lvl}"
    return f"{loc}:{pat}"

def make_correction_pair(row):
    orig = row["원 형태소"] if row["원 형태소"] != "0" else "∅"
    orig_tag = row["형태 주석"] if row["형태 주석"] != "0" else ""
    corr = row["교정 형태소"] if row["교정 형태소"] != "0" else "∅"
    corr_tag = row["교정 주석"] if row["교정 주석"] != "0" else ""
    orig_str = f"{orig}/{orig_tag}" if orig_tag else orig
    corr_str = f"{corr}/{corr_tag}" if corr_tag else corr
    return f"{orig_str}→{corr_str}"

In [5]:
err_rows = df[df["오류 양상"] != "0"].copy()
err_rows["signature"] = err_rows.apply(make_error_signature, axis=1)
err_rows["correction_pair"] = err_rows.apply(make_correction_pair, axis=1)

In [10]:
sentence_errors = err_rows.groupby(["표본 번호", "문장"]).agg(
    error_signatures=("signature", list),
    correction_pairs=("correction_pair", list),
    error_patterns=("오류 양상", list),
).reset_index()

In [14]:
sentence_errors[sentence_errors['문장'] == "지난주에 제 친구랑 같이 뮤치칼에 가 봤다."]

,표본 번호,문장,error_signatures,correction_pairs,error_patterns
34165,9685,지난주에 제 친구랑 같이 뮤치칼에 가 봤다.,"[CNP:ADD;REP:SH, CNNG:MIF, CNNG:OM]","[저/NP→ADD/NP, 뮤치칼/NNG→뮤지컬/NNG, ∅→공연/NNG]","[ADD;REP, MIF, OM]"


In [15]:
df[df["표본 번호"] == 9685]

,표본 번호,문장,어절 번호,원 어절,형태 번호,원 형태소,형태 주석,교정 형태소,교정 주석,분석 불가능,...,한국어 등급,국적,교포 여부,거주 기간,학습 기간,학습 목표,모국어,기타 언어 1,기타 언어 2,기타 언어 3
1067684,9685,약속 날,1,약속,1,약속,NNG,0,0,0,...,2급,미국,외국인,4개월,24개월,거주,영어,아르메니아어,NaN,NaN
1067685,9685,약속 날,2,날,2,날,NNG,0,0,0,...,2급,미국,외국인,4개월,24개월,거주,영어,아르메니아어,NaN,NaN
1067686,9685,지난주에 제 친구랑 같이 뮤치칼에 가 봤다.,1,지난주에,1,지난주,NNG,0,0,0,...,2급,미국,외국인,4개월,24개월,거주,영어,아르메니아어,NaN,NaN
1067687,9685,지난주에 제 친구랑 같이 뮤치칼에 가 봤다.,1,지난주에,2,에,JKB,0,0,0,...,2급,미국,외국인,4개월,24개월,거주,영어,아르메니아어,NaN,NaN
1067688,9685,지난주에 제 친구랑 같이 뮤치칼에 가 봤다.,2,제,3,저,NP,ADD,NP,0,...,2급,미국,외국인,4개월,24개월,거주,영어,아르메니아어,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067893,9685,제 친구 함께 얼마나 행복한지 모르다.,5,행복한지,9,0,0,았,EP,0,...,2급,미국,외국인,4개월,24개월,거주,영어,아르메니아어,NaN,NaN
1067894,9685,제 친구 함께 얼마나 행복한지 모르다.,5,행복한지,10,ㄴ지,EF,는지,EF,0,...,2급,미국,외국인,4개월,24개월,거주,영어,아르메니아어,NaN,NaN
1067895,9685,제 친구 함께 얼마나 행복한지 모르다.,6,모르다.,11,모르,VV,0,0,0,...,2급,미국,외국인,4개월,24개월,거주,영어,아르메니아어,NaN,NaN
1067896,9685,제 친구 함께 얼마나 행복한지 모르다.,6,모르다.,12,다,EF,ㄴ다,EF,0,...,2급,미국,외국인,4개월,24개월,거주,영어,아르메니아어,NaN,NaN
